# AES (Automated Essay Scoring) — Fine-tuning Llama-3.2-3B dengan QLoRA (Unsloth)

Notebook ini menggabungkan 4 script menjadi satu pipeline end-to-end yang bisa dijalankan
di JupyterLab dengan GPU:

| Bagian | Sumber asli | Fungsi |
|---|---|---|
| 0 | — | Setup environment, cek GPU, instalasi dependency |
| 1 | `01_prepare_dataset.py` | Load Excel -> cleaning -> stratified split -> `train/val/test.jsonl` |
| 2 | `02_train_lora_unsloth.py` | QLoRA fine-tuning Llama-3.2-3B-Instruct via Unsloth |
| 3A | `03_evaluate.py` (dimodifikasi) | Evaluasi **langsung di notebook** (model LoRA di memori GPU, tanpa Ollama) |
| 3B | `03_evaluate.py` (asli) | Evaluasi **via Ollama API** (setelah export GGUF + `ollama create`) |
| 4 | `export_aes_dataset.py` | Referensi saja (Django management command, **tidak dieksekusi** di notebook ini) |

## Prasyarat
- **GPU wajib** untuk Bagian 2 (training) dan Bagian 3A (inference in-memory). Minimal 8GB VRAM
  (RTX 3060 / T4 Colab / Kaggle) untuk Llama-3.2-3B dengan 4-bit quantization.
- Bagian 1 (data prep) bisa jalan di CPU saja.
- Bagian 3B butuh [Ollama](https://ollama.com) ter-install & running secara terpisah di mesin yang sama.
- Letakkan `Dataset_Machine_Learning.xlsx` di `./data/` sebelum menjalankan Bagian 1.

## Struktur folder yang diasumsikan
```
aes_pipeline/                      <- jalankan notebook ini dari sini
├── aes_finetuning_pipeline.ipynb
├── data/
│   ├── Dataset_Machine_Learning.xlsx   <- taruh manual di sini
│   ├── cleaned_dataset.csv             <- output Bagian 1 (isi kolom 'rubrik' manual!)
│   ├── train.jsonl / val.jsonl / test.jsonl
└── outputs/
    ├── llama32_3b_aes_lora/       <- adapter LoRA
    └── llama32_3b_aes_gguf/       <- hasil export GGUF (untuk Ollama)
```

> ⚠️ **Catatan penting dari script asli**: kolom `rubrik` di `cleaned_dataset.csv` sengaja
> dikosongkan dan **wajib diisi manual** (3-5 poin kunci per soal) sebelum training final.
> Tanpa rubrik, fine-tuning hanya mengajarkan gaya/format penilaian, bukan kriteria akademik
> — model tidak akan benar-benar belajar *menilai*, hanya belajar meniru format skor.


## Bagian 0 — Setup Environment

In [ ]:
# Cek GPU tersedia — WAJIB sebelum lanjut ke Bagian 2 & 3A
import subprocess
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM total: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print("!! Tidak ada GPU terdeteksi. Bagian 2 (training) dan 3A (inference) tidak akan jalan.")

try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print("nvidia-smi tidak ditemukan.")


In [ ]:
# Instalasi dependency (jalankan SEKALI SAJA per environment/kernel)
# Sesuaikan tag [cu121-torch230] dengan versi CUDA/torch di mesin kamu --
# cek https://github.com/unslothai/unsloth untuk kombinasi yang cocok saat ini.

# %pip install "unsloth[cu121-torch230] @ git+https://github.com/unslothai/unsloth.git"
# %pip install --no-deps trl peft accelerate bitsandbytes
# %pip install pandas openpyxl scikit-learn matplotlib seaborn requests


In [ ]:
from __future__ import annotations

import json
import re
import logging
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score, confusion_matrix, classification_report

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)
sns.set_theme(style="whitegrid")

# ---- Path project (sesuaikan jika struktur folder kamu berbeda) ----
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = DATA_DIR / "Dataset_Machine_Learning.xlsx"

RANDOM_SEED = 42
VALID_SCORES = {0.0, 5.0, 10.0}  # skala penilaian resmi sesuai kolom 'Skor (0/5/10)'

SYSTEM_PROMPT = (
    "Anda adalah asisten penilai esai untuk mata kuliah Machine Learning. "
    "Nilai jawaban mahasiswa HANYA berdasarkan rubrik yang diberikan. "
    "Berikan skor (0, 5, atau 10) dan alasan singkat dalam Bahasa Indonesia. "
    "Format jawaban WAJIB JSON: {\"skor\": <0|5|10>, \"alasan\": \"...\"}"
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR    :", DATA_DIR)
print("OUTPUT_DIR  :", OUTPUT_DIR)


## Bagian 1 — Data Preparation (`01_prepare_dataset.py`)

Load sheet `Soal` + `Jawaban` dari Excel, bersihkan, gabungkan, lalu split stratified
per `(Kode Soal x Skor)` supaya distribusi kelas skor terwakili proporsional di
train/val/test — ini penting agar evaluasi QWK di Bagian 3 valid (bukan bias ke satu subset).

In [ ]:
def clean_text(text: object) -> str | None:
    '''Normalisasi teks jawaban: rapikan newline literal, whitespace, dan string kosong.'''
    if not isinstance(text, str):
        return None
    text = text.replace("\\n", "\n").strip()
    text = re.sub(r"\n{2,}", "\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    if text in {"", "-", "NaN", "nan"}:
        return None
    return text


def load_and_merge(path: Path) -> pd.DataFrame:
    '''Load kedua sheet, join by Kode Soal, dan buang baris yang tidak lengkap.'''
    if not path.exists():
        raise FileNotFoundError(f"File tidak ditemukan: {path}")

    soal = pd.read_excel(path, sheet_name="Soal")
    jawaban = pd.read_excel(path, sheet_name="Jawaban")

    n_before = len(jawaban)

    jawaban["Jawaban"] = jawaban["Jawaban"].apply(clean_text)
    jawaban = jawaban.dropna(subset=["Kode Soal", "Jawaban", "Skor (0/5/10)"])
    jawaban = jawaban[jawaban["Skor (0/5/10)"].isin(VALID_SCORES)]  # buang outlier skor (mis. 3.0)
    jawaban["Kode Soal"] = jawaban["Kode Soal"].astype(int)
    jawaban = jawaban.drop_duplicates(subset=["Kode Soal", "Jawaban"])

    df = jawaban.merge(soal, left_on="Kode Soal", right_on="Kode Soal", how="left")
    df = df.rename(columns={"Soal": "pertanyaan", "Jawaban": "jawaban", "Skor (0/5/10)": "skor"})
    df["skor"] = df["skor"].astype(int)
    df["rubrik"] = ""  # placeholder wajib diisi manual sebelum training final

    n_after = len(df)
    logger.info(
        "Baris awal: %d | Baris valid setelah cleaning: %d | Dibuang: %d (%.1f%%)",
        n_before, n_after, n_before - n_after, 100 * (n_before - n_after) / n_before,
    )
    logger.info("Distribusi skor final:\n%s", df["skor"].value_counts().sort_index())
    logger.info("Distribusi soal final:\n%s", df["Kode Soal"].value_counts().sort_index())

    return df.reset_index(drop=True)


In [ ]:
df = load_and_merge(INPUT_PATH)
df.head()

### EDA singkat — distribusi skor & soal

Cek keseimbangan kelas sebelum split. Kelas yang sangat timpang (mis. skor `5` dominan)
berisiko membuat model bias memprediksi kelas mayoritas — perlu diperhatikan saat membaca
QWK & classification report di Bagian 3.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.countplot(data=df, x="skor", order=[0, 5, 10], ax=axes[0], palette="viridis")
axes[0].set_title("Distribusi Skor")
axes[0].set_xlabel("Skor")
axes[0].set_ylabel("Jumlah jawaban")

sns.countplot(data=df, x="Kode Soal", ax=axes[1], palette="mako")
axes[1].set_title("Distribusi Jumlah Jawaban per Soal")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

print(pd.crosstab(df["Kode Soal"], df["skor"]))


In [ ]:
df.to_csv(DATA_DIR / "cleaned_dataset.csv", index=False)
logger.info("cleaned_dataset.csv disimpan di %s", DATA_DIR / "cleaned_dataset.csv")
print("\n>>> ISI KOLOM 'rubrik' DI cleaned_dataset.csv SECARA MANUAL SEBELUM TRAINING FINAL <<<")
print("Tanpa rubrik, model hanya belajar format skor, bukan kriteria penilaian akademik.")


> **Langkah manual di sini**: buka `data/cleaned_dataset.csv`, isi kolom `rubrik` untuk
> setiap `Kode Soal` (3-5 poin kunci penilaian), simpan kembali sebagai `cleaned_dataset.csv`
> di folder yang sama. Baru lanjutkan ke sel berikut, yang akan **membaca ulang file yang
> sudah diisi rubrik**.

In [ ]:
# Baca ulang cleaned_dataset.csv (setelah kolom 'rubrik' diisi manual)
df = pd.read_csv(DATA_DIR / "cleaned_dataset.csv")
df["rubrik"] = df["rubrik"].fillna("")
n_missing_rubrik = (df["rubrik"].str.strip() == "").sum()
if n_missing_rubrik:
    logger.warning(
        "%d/%d baris masih belum ada rubrik. Model akan menggunakan placeholder untuk baris ini.",
        n_missing_rubrik, len(df),
    )
df.head()


In [ ]:
def stratified_split(
    df: pd.DataFrame, val_size: float = 0.15, test_size: float = 0.15
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    '''Split stratified pada gabungan Kode Soal + skor agar distribusi tiap subset seimbang.'''
    strata = df["Kode Soal"].astype(str) + "_" + df["skor"].astype(str)

    # buang strata dengan <2 anggota (tidak bisa distratifikasi)
    counts = strata.value_counts()
    valid_idx = strata.isin(counts[counts >= 2].index)
    dropped = (~valid_idx).sum()
    if dropped:
        logger.warning("Membuang %d baris dari strata langka (<2 anggota) sebelum split.", dropped)
    df, strata = df[valid_idx].reset_index(drop=True), strata[valid_idx].reset_index(drop=True)

    train_df, temp_df, strata_train, strata_temp = train_test_split(
        df, strata, test_size=val_size + test_size, stratify=strata, random_state=RANDOM_SEED
    )
    # re-check strata temp untuk split kedua
    temp_counts = strata_temp.value_counts()
    valid_temp = strata_temp.isin(temp_counts[temp_counts >= 2].index)
    temp_df, strata_temp = temp_df[valid_temp], strata_temp[valid_temp]

    rel_test = test_size / (val_size + test_size)
    val_df, test_df = train_test_split(
        temp_df, test_size=rel_test, stratify=strata_temp, random_state=RANDOM_SEED
    )
    return train_df, val_df, test_df


def to_chat_jsonl(df: pd.DataFrame, path: Path) -> None:
    '''Tulis dataset dalam format chat SFT (compatible Llama-3.2 instruct template).'''
    with path.open("w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            rubrik = row["rubrik"] or "(rubrik belum diisi - lengkapi sebelum training final)"
            user_msg = (
                f"Pertanyaan: {row['pertanyaan']}\n"
                f"Rubrik penilaian: {rubrik}\n"
                f"Jawaban mahasiswa: {row['jawaban']}\n\n"
                "Berikan skor dan alasan dalam format JSON."
            )
            assistant_msg = json.dumps(
                {"skor": int(row["skor"]), "alasan": ""}, ensure_ascii=False
            )
            record = {
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg},
                    {"role": "assistant", "content": assistant_msg},
                ]
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    logger.info("Ditulis %d baris -> %s", len(df), path)


In [ ]:
train_df, val_df, test_df = stratified_split(df)
logger.info("Split -> train: %d | val: %d | test: %d", len(train_df), len(val_df), len(test_df))

to_chat_jsonl(train_df, DATA_DIR / "train.jsonl")
to_chat_jsonl(val_df, DATA_DIR / "val.jsonl")
to_chat_jsonl(test_df, DATA_DIR / "test.jsonl")


In [ ]:
# Sanity check: distribusi skor di tiap split harus proporsional (bukti stratifikasi bekerja)
split_dist = pd.concat(
    {
        "train": train_df["skor"].value_counts(normalize=True).sort_index(),
        "val": val_df["skor"].value_counts(normalize=True).sort_index(),
        "test": test_df["skor"].value_counts(normalize=True).sort_index(),
    },
    axis=1,
)
split_dist.plot(kind="bar", figsize=(7, 4), title="Proporsi Skor per Split")
plt.ylabel("Proporsi")
plt.xlabel("Skor")
plt.tight_layout()
plt.show()
split_dist


## Bagian 2 — Fine-tuning QLoRA dengan Unsloth (`02_train_lora_unsloth.py`)

> **Butuh GPU** (min. 8GB VRAM). Jika sel Bagian 0 di atas melaporkan "Tidak ada GPU
> terdeteksi", hentikan di sini dan pindah ke environment ber-GPU (mis. Colab/Kaggle/RTX lokal).

In [ ]:
# ---- Hyperparameters (dokumentasikan semua untuk reproducibility) ----
BASE_MODEL = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"  # quantized, hemat VRAM
MAX_SEQ_LENGTH = 2048
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0  # 0 = optimized path di Unsloth
LEARNING_RATE = 2e-4
NUM_TRAIN_EPOCHS = 3          # dataset kecil (~300 baris) -> awasi overfitting, cek val loss tiap epoch
PER_DEVICE_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4          # effective batch size = 8
SEED = 42

LORA_OUTPUT_DIR = OUTPUT_DIR / "llama32_3b_aes_lora"
GGUF_OUTPUT_DIR = OUTPUT_DIR / "llama32_3b_aes_gguf"


In [ ]:
from unsloth import FastLanguageModel

logger.info("Loading base model: %s", BASE_MODEL)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)


In [ ]:
from datasets import load_dataset

train_path = DATA_DIR / "train.jsonl"
val_path = DATA_DIR / "val.jsonl"
if not train_path.exists() or not val_path.exists():
    raise FileNotFoundError(f"Dataset tidak ditemukan di {DATA_DIR}. Jalankan Bagian 1 dulu.")

dataset = load_dataset(
    "json", data_files={"train": str(train_path), "validation": str(val_path)}
)

def format_chat(example: dict) -> dict:
    # apply_chat_template menyusun prompt sesuai template resmi Llama-3.2-Instruct
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

dataset = dataset.map(format_chat, remove_columns=dataset["train"].column_names)
print(dataset)
print("\n--- Contoh formatted text ---\n")
print(dataset["train"][0]["text"][:800])


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=SFTConfig(
        output_dir=str(LORA_OUTPUT_DIR),
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        learning_rate=LEARNING_RATE,
        eval_strategy="epoch",       # pantau val loss tiap epoch -- stop kalau naik (overfit)
        save_strategy="epoch",
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=SEED,
        report_to="none",
    ),
)

logger.info("Mulai training...")
train_result = trainer.train()


In [ ]:
# Plot train/eval loss per epoch dari log_history — cek overfitting sebelum lanjut ke evaluasi
history = pd.DataFrame(trainer.state.log_history)
train_loss = history.dropna(subset=["loss"])[["epoch", "loss"]] if "loss" in history else None
eval_loss = history.dropna(subset=["eval_loss"])[["epoch", "eval_loss"]] if "eval_loss" in history else None

plt.figure(figsize=(7, 4))
if train_loss is not None and not train_loss.empty:
    plt.plot(train_loss["epoch"], train_loss["loss"], label="train_loss", marker="o")
if eval_loss is not None and not eval_loss.empty:
    plt.plot(eval_loss["epoch"], eval_loss["eval_loss"], label="eval_loss", marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
logger.info("Menyimpan adapter LoRA + tokenizer ke %s", LORA_OUTPUT_DIR)
model.save_pretrained(str(LORA_OUTPUT_DIR))
tokenizer.save_pretrained(str(LORA_OUTPUT_DIR))

# Ekspor ke GGUF (q4_k_m) -- HANYA perlu jika kamu mau pakai jalur evaluasi 3B (Ollama)
logger.info("Mengekspor ke GGUF (q4_k_m) di %s", GGUF_OUTPUT_DIR)
model.save_pretrained_gguf(
    str(GGUF_OUTPUT_DIR), tokenizer, quantization_method="q4_k_m"
)
logger.info("Selesai. File .gguf ada di %s -- lanjut ke Modelfile Ollama jika perlu (Bagian 3B).", GGUF_OUTPUT_DIR)


## Bagian 3A — Evaluasi langsung di notebook (in-memory, tanpa Ollama)

Menggunakan model hasil fine-tuning yang masih ada di GPU memory (`model`, `tokenizer` dari
Bagian 2) untuk generate prediksi langsung, tanpa perlu export GGUF + serve Ollama.
Metrik utama tetap **Quadratic Weighted Kappa (QWK)** karena skor bersifat ordinal
(0 < 5 < 10) — accuracy saja menyembunyikan seberapa jauh model "meleset".

In [ ]:
FastLanguageModel.for_inference(model)  # aktifkan mode inference (2x lebih cepat di Unsloth)

def load_test_set(path: Path) -> list[dict]:
    if not path.exists():
        raise FileNotFoundError(f"{path} tidak ditemukan -- jalankan Bagian 1 dulu.")
    records = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))
    return records


def predict_score_inmemory(system_prompt: str, user_prompt: str, max_new_tokens: int = 128) -> int | None:
    '''Generate respons dari model in-memory, parse skor. Return None jika gagal parse.'''
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,      # deterministik untuk evaluasi
            temperature=None,
            top_p=None,
            top_k=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(output_ids[0][inputs.shape[-1]:], skip_special_tokens=True)

    match = re.search(r'"skor"\s*:\s*(\d+)', generated)
    if not match:
        logger.warning("Tidak bisa parse skor dari respons: %r", generated[:200])
        return None

    score = int(match.group(1))
    if score not in {0, 5, 10}:
        logger.warning("Skor di luar skala valid: %d", score)
        return None
    return score


In [ ]:
records = load_test_set(DATA_DIR / "test.jsonl")
logger.info("Mengevaluasi %d contoh dari test set (in-memory)...", len(records))

y_true, y_pred = [], []
n_failed = 0

for i, rec in enumerate(records):
    system_msg = rec["messages"][0]["content"]
    user_msg = rec["messages"][1]["content"]
    true_score = json.loads(rec["messages"][2]["content"])["skor"]

    pred_score = predict_score_inmemory(system_msg, user_msg)
    if pred_score is None:
        n_failed += 1
        continue

    y_true.append(true_score)
    y_pred.append(pred_score)

    if (i + 1) % 10 == 0:
        logger.info("Progress: %d/%d", i + 1, len(records))

if n_failed:
    logger.warning("%d/%d prediksi gagal diparse dan dikecualikan dari metrik.", n_failed, len(records))


In [ ]:
VALID_SCORES = [0, 5, 10]

if not y_true:
    logger.error("Tidak ada prediksi valid.")
else:
    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    print(f"Quadratic Weighted Kappa: {qwk:.4f}")
    print("Interpretasi kasar QWK: <0.4 lemah | 0.4-0.6 cukup | 0.6-0.8 baik | >0.8 sangat baik")

    cm = confusion_matrix(y_true, y_pred, labels=VALID_SCORES)
    plt.figure(figsize=(4.5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=VALID_SCORES, yticklabels=VALID_SCORES)
    plt.xlabel("Prediksi")
    plt.ylabel("Skor Asli")
    plt.title(f"Confusion Matrix (QWK={qwk:.3f})")
    plt.tight_layout()
    plt.show()

    print(classification_report(y_true, y_pred, labels=VALID_SCORES))


## Bagian 3B — Evaluasi via Ollama API (`03_evaluate.py` asli, opsional)

Jalur ini sesuai script asli: model di-export ke GGUF (sudah dilakukan di Bagian 2),
lalu di-serve lewat Ollama, baru dipanggil via HTTP API. Berguna untuk membandingkan hasil
model **setelah** quantization GGUF (bisa sedikit berbeda dari hasil in-memory di Bagian 3A)
atau untuk deployment terpisah dari notebook training.

**Prasyarat (jalankan di terminal, di luar notebook):**
```bash
# 1. Install Ollama: https://ollama.com/download
# 2. Buat Modelfile yang menunjuk ke GGUF hasil Bagian 2
cat > Modelfile << 'EOF'
FROM ./outputs/llama32_3b_aes_gguf/model-q4_k_m.gguf
EOF

# 3. Register model ke Ollama
ollama create llama32-aes -f Modelfile

# 4. Pastikan Ollama server jalan (biasanya otomatis sebagai service)
ollama serve
```

In [ ]:
import requests

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL_NAME = "llama32-aes"  # sesuaikan dengan nama hasil `ollama create` di atas


def predict_score_ollama(system_prompt: str, user_prompt: str) -> int | None:
    '''Panggil Ollama, parse skor dari respons JSON. Return None jika gagal parse.'''
    try:
        resp = requests.post(
            OLLAMA_URL,
            json={
                "model": MODEL_NAME,
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                "stream": False,
                "options": {"temperature": 0.0},  # deterministik untuk evaluasi
            },
            timeout=60,
        )
        resp.raise_for_status()
        content = resp.json()["message"]["content"]
    except (requests.RequestException, KeyError) as exc:
        logger.warning("Gagal memanggil Ollama: %s", exc)
        return None

    match = re.search(r'"skor"\s*:\s*(\d+)', content)
    if not match:
        logger.warning("Tidak bisa parse skor dari respons: %r", content[:200])
        return None

    score = int(match.group(1))
    if score not in {0, 5, 10}:
        logger.warning("Skor di luar skala valid: %d", score)
        return None
    return score


In [ ]:
# Sanity check koneksi Ollama sebelum evaluasi penuh
try:
    _ping = requests.get("http://localhost:11434", timeout=3)
    print("Ollama server terdeteksi:", _ping.status_code)
except requests.RequestException as exc:
    print("!! Ollama server tidak terdeteksi di localhost:11434 -- jalankan `ollama serve` dulu.")
    print("Error:", exc)


In [ ]:
records = load_test_set(DATA_DIR / "test.jsonl")
logger.info("Mengevaluasi %d contoh dari test set (via Ollama)...", len(records))

y_true_ol, y_pred_ol = [], []
n_failed_ol = 0

for i, rec in enumerate(records):
    system_msg = rec["messages"][0]["content"]
    user_msg = rec["messages"][1]["content"]
    true_score = json.loads(rec["messages"][2]["content"])["skor"]

    pred_score = predict_score_ollama(system_msg, user_msg)
    if pred_score is None:
        n_failed_ol += 1
        continue

    y_true_ol.append(true_score)
    y_pred_ol.append(pred_score)

    if (i + 1) % 10 == 0:
        logger.info("Progress: %d/%d", i + 1, len(records))

if n_failed_ol:
    logger.warning("%d/%d prediksi gagal diparse dan dikecualikan dari metrik.", n_failed_ol, len(records))


In [ ]:
if not y_true_ol:
    logger.error("Tidak ada prediksi valid -- cek koneksi Ollama / nama model.")
else:
    qwk_ol = cohen_kappa_score(y_true_ol, y_pred_ol, weights="quadratic")
    print(f"Quadratic Weighted Kappa (Ollama/GGUF): {qwk_ol:.4f}")

    cm_ol = confusion_matrix(y_true_ol, y_pred_ol, labels=VALID_SCORES)
    plt.figure(figsize=(4.5, 4))
    sns.heatmap(cm_ol, annot=True, fmt="d", cmap="Oranges",
                xticklabels=VALID_SCORES, yticklabels=VALID_SCORES)
    plt.xlabel("Prediksi")
    plt.ylabel("Skor Asli")
    plt.title(f"Confusion Matrix Ollama (QWK={qwk_ol:.3f})")
    plt.tight_layout()
    plt.show()

    print(classification_report(y_true_ol, y_pred_ol, labels=VALID_SCORES))


### Membandingkan 3A vs 3B

Jika QWK di Bagian 3A (in-memory, bf16/4-bit training precision) berbeda jauh dari
Bagian 3B (Ollama/GGUF q4_k_m), itu indikasi **degradasi akibat quantization GGUF** —
pertimbangkan quantization method lain (mis. `q8_0`) jika selisihnya signifikan untuk
use case kamu.

## Bagian 4 — `export_aes_dataset.py` (referensi saja, **tidak dieksekusi** di notebook ini)

Script ini adalah **Django management command**, bukan bagian dari alur training/evaluasi
notebook — ia butuh konteks aplikasi Django penuh (`apps.exams.models.Soal`,
`apps.submissions.models.Jawaban`, `manage.py`, dsb.) yang tidak tersedia di sini.

Fungsinya: menggantikan alur export Excel manual (Bagian 1) dengan export langsung dari
database production begitu sistem AES sudah live, dengan format output `train/val/test.jsonl`
yang **kompatibel** dengan Bagian 2 (skema `messages` sama persis).

**Lokasi file di project Django**: `apps/submissions/management/commands/export_aes_dataset.py`

**Cara pakai (di project Django, bukan di notebook ini):**
```bash
python manage.py export_aes_dataset --output ./aes_export --only-verified
```

Opsi:
- `--output` — folder output (default: `./aes_export`)
- `--only-verified` — hanya ambil baris yang sudah divalidasi dosen. **Disarankan** dipakai
  begitu field verifikasi ditambahkan ke model `Jawaban`; tanpa flag ini, semua baris
  `grading_status='done'` ikut ter-export apa adanya (termasuk hasil AI yang belum divalidasi
  manusia — risiko *label leakage* dari model lama ke model baru).

Kode lengkapnya tidak diduplikasi di sini karena tidak bisa dijalankan di lingkungan notebook;
lihat file asli `export_aes_dataset.py` di project Django kamu. Setelah dijalankan, hasil
`train.jsonl` / `val.jsonl` / `test.jsonl`-nya bisa langsung dipakai menggantikan output
Bagian 1 sebagai input ke Bagian 2 (cukup arahkan `DATA_DIR` di Bagian 0 ke folder
`--output` tersebut).